### Introduction To Data Ingestion

In [18]:
import os
from typing import List, Dict, Any
import pandas as pd

In [19]:
#Langchain Document structure
from langchain_core.documents import Document
#For Document Splitter
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    TokenTextSplitter
)
print("Setup complete")

Setup complete


### Understanding Document Structure In Langchain

In [20]:
# Create a simple document
doc = Document(
    page_content="This is the main text content that will be embedded and searched.",
    metadata={
        "source": "example.txt",
        "page": 1,
        "author": "John Doe",
        "date_created": "2023-10-01",
        "custom_field": "any_value"
    }
)
print("Document Structure")
print(f"Content: {doc.page_content}")
print(f"Metadata: {doc.metadata}")

Document Structure
Content: This is the main text content that will be embedded and searched.
Metadata: {'source': 'example.txt', 'page': 1, 'author': 'John Doe', 'date_created': '2023-10-01', 'custom_field': 'any_value'}


In [21]:
type(doc)

langchain_core.documents.base.Document

### Reading a Text files (.txt)

In [22]:
# Create a simple txt file
import os
os.makedirs("data/text_files", exist_ok=True)
print(os.path.abspath("data/text_files"))

c:\RAG\DataIngestionParsing\data\text_files


In [23]:
sample_texts={
    "data/text_files/python_intro.txt": """Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",
    "data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems"""
}

for filepath,content in sample_texts.items():
    with open(filepath, "w") as f:
        f.write(content)    

print("Sample text files created successfully.")

Sample text files created successfully.


### TextLoader - Read Single File

In [24]:
from langchain_community.document_loaders import TextLoader

# Loading a Single Text File
loader=TextLoader("data/text_files/python_intro.txt", encoding="utf-8")

documents=loader.load()
print(type(documents))
print(documents)

<class 'list'>
[Document(metadata={'source': 'data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.')]


In [25]:
print(f"Number of documents loaded: {len(documents)}")
# Display first 100 characters of page_content
print(f"Content of the first document:\n{documents[0].page_content[:100]}")
print(f"Metadata of the first document: {documents[0].metadata}")

Number of documents loaded: 1
Content of the first document:
Python Programming Introduction

Python is a high-level, interpreted programming language known for 
Metadata of the first document: {'source': 'data/text_files/python_intro.txt'}


### DirectoryLoader - Multiple Text Files

In [26]:
from langchain_community.document_loaders import DirectoryLoader

# load all the text files from the directory
dir_loader = DirectoryLoader("data/text_files", 
                    glob="**/*.txt", ## Pattern to match files
                    loader_cls= TextLoader, ## loader class to use
                    loader_kwargs={'encoding': 'utf-8'},
                    show_progress=True
)

documents=dir_loader.load()

print(f"Number of documents loaded: {len(documents)}")
for i, doc in enumerate(documents):
    print(f"Document {i+1}:")
    print(f"Content: {doc.page_content[:100]}")
    print(f"Length: {len(doc.page_content)} characters")
    print(f"Metadata: {doc.metadata}")

100%|██████████| 2/2 [00:00<00:00, 399.97it/s]

Number of documents loaded: 2
Document 1:
Content: Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables system
Length: 568 characters
Metadata: {'source': 'data\\text_files\\machine_learning.txt'}
Document 2:
Content: Python Programming Introduction

Python is a high-level, interpreted programming language known for 
Length: 489 characters
Metadata: {'source': 'data\\text_files\\python_intro.txt'}


### Text Splitting Strategies

In [27]:
# Different Text Splitting Strategies
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    TokenTextSplitter
)

print(documents)

[Document(metadata={'source': 'data\\text_files\\machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems'), Document(metadata={'source': 'data\\text_files\\python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming la

### Method  - CharacterTextSplitter

In [28]:
text = documents[0].page_content
print(text)
print("-----")
char_splitter = CharacterTextSplitter(
    separator="\n", #Split on new lines
    chunk_size=200, #Max chunk size in characters
    chunk_overlap=20, #Overlap between chunks in characters
    length_function=len # How to measure chunk size
)

char_chunks = char_splitter.split_text(text)    
print(f"Created {len(char_chunks)} chunks")
print(f"First chunk: {char_chunks[0][:100]}...")

Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems
-----
Created 4 chunks
First chunk: Machine Learning Basics
Machine learning is a subset of artificial intelligence that enables systems...


In [29]:
print(char_chunks[0])
print("-------------")
print(char_chunks[1])
print("-------------")
print(char_chunks[2])
print("-------------")
print(char_chunks[3])
print("-------------")

Machine Learning Basics
Machine learning is a subset of artificial intelligence that enables systems to learn and improve
-------------
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.
Types of Machine Learning:
-------------
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties
-------------
Applications include image recognition, speech processing, and recommendation systems
-------------


### Method  - RecursiveCharacterTextSplitter

In [30]:
print(text)
print("-----")
recursive_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ""], #Try these separators in order
    chunk_size=200,
    chunk_overlap=20,
    length_function=len
)

recursive_chunks = recursive_splitter.split_text(text)
print(f"Created {len(recursive_chunks)} chunks")
print(f"First chunk: {recursive_chunks[0][:100]}...")

Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems
-----
Created 6 chunks
First chunk: Machine Learning Basics...


In [31]:
print(recursive_chunks[0])
print("-------------")
print(recursive_chunks[1])
print("-------------")
print(recursive_chunks[2])
print("-------------")
print(recursive_chunks[3])
print("-------------")
print(recursive_chunks[4])
print("-------------")
print(recursive_chunks[5])
print("-------------")

Machine Learning Basics
-------------
Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
-------------
that can access data and use it to learn for themselves.
-------------
Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
-------------
3. Reinforcement Learning: Learning through rewards and penalties
-------------
Applications include image recognition, speech processing, and recommendation systems
-------------


##### To showcase that text overlap is happening (with one separator)

In [32]:
# Create text without natural break points
simple_text = "This is sentence one and it is quite long. This is sentence two and it is also quite long. This is sentence three this is very long."

splitter = RecursiveCharacterTextSplitter(
    separators=[" "], #Only split on spaces
    chunk_size=80,
    chunk_overlap=20,
    length_function=len
)

chunks = splitter.split_text(simple_text)

print(f"\nSimple text example - {len(chunks)} chunks: \n")

for i in range(len(chunks) - 1):
    print(f"Chunk {i+1}: '{chunks[i]}'")
    print(f"Chunk {i+2}: '{chunks[i+1]}")


Simple text example - 2 chunks: 

Chunk 1: 'This is sentence one and it is quite long. This is sentence two and it is also'
Chunk 2: 'two and it is also quite long. This is sentence three this is very long.


### Method  - Token-based splitting (not only focus on character it will focus on space)

In [33]:
token_splitter = TokenTextSplitter(
    chunk_size=50, # Size in tokens (not characters)
    chunk_overlap=10
)

token_chunks = token_splitter.split_text(text)
print(f"Created {len(token_chunks)} chunks")
print(f"First Chunk: {token_chunks[0][:100]}...")

Created 3 chunks
First Chunk: Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables system...
